### 修改参数运行Monod，输出到处理目录

In [3]:
import monod
from monod import *
from monod.preprocess import *
from monod.extract_data import *
from monod.cme_toolbox import CMEModel
from monod import inference, mminference
from monod.analysis import *
from monod import preprocess, extract_data, cme_toolbox, inference, analysis
import logging, sys
logging.basicConfig(stream=sys.stdout)
log = logging.getLogger()
log.setLevel(logging.INFO)
import warnings
warnings.filterwarnings("ignore") #warning suppression within script is not respected by colab
warnings.simplefilter('ignore')
import matplotlib.pyplot as plt

In [12]:
### 输入loom文件目录  CD4, CD8, MNP, NK, DC, B
# dataset_names = ['covid1', 'covid2', 'covid3', 'covid4', 'covid5', 'covid6', 'covid7', 'covid8', 'covid9', 'covid10', 'covid11']   ### 改文件名 
# dataset_names = ['flu1', 'flu2', 'flu3', 'flu4', 'flu5']   ### 改文件名 
dataset_names = ['nor1', 'nor2', 'nor3', 'nor4']   ### 改文件名 
loom_filepaths = ['/data/wuwq/noise/Monod_FLU/symbol_loom/B/'+x+'.loom' for x in dataset_names]   ### 改文件夹名
transcriptome_filepath = '/data2/wuwq/noise/monod_FLU/polyA_ref/gg_200525_genome_polyA_cum_3'  #不变
attribute_names=[('nascent','mature'),'Gene','barcode']  #不变

In [ ]:
### 输出目录 'Bursty' or 'Extrinsic'  ### 基因数可以变
import matplotlib.pyplot as plt
plt.close('all')  # 关闭所有现有 figure，避免残留
##### 'Bursty'
dir_string,dataset_strings = monod.preprocess.construct_batch(loom_filepaths, \
                                             transcriptome_filepath, \
                                             dataset_names, \
                                             attribute_names=attribute_names,\
                                             batch_location='/data/wuwq/noise/Monod_FLU/OUT', meta='nor',batch_id= 'B_Bursty',\
                                             n_genes=1000) # 可以改基因数： 250 or 5,000  → 1000足够了
n_datasets = len(dataset_names)
lb = [-1.0, -1.8, -1.8 ]
ub = [4.2, 2.5, 3.5]
grid = [6,7]
# 参数选择：
# biological_model = {'Bursty','Constitutive','Extrinsic','CIR'}       sequencing_model = {'None','Bernoulli','Poisson'}
result_strings = []
for i in range(n_datasets):
    fitmodel = monod.cme_toolbox.CMEModel('Bursty','Poisson') 
    inference_parameters = monod.inference.InferenceParameters(lb,ub,[-8, -3],[-5, 0],grid,\
                dataset_strings[i],fitmodel,use_lengths = True,
                gradient_params = {'max_iterations':5,'init_pattern':'moments','num_restarts':1})
    search_data = monod.extract_data.extract_data(loom_filepaths[i], transcriptome_filepath, dataset_names[i],
                        dataset_strings[i], dir_string, dataset_attr_names=attribute_names)
    full_result_string = inference_parameters.fit_all_grid_points(1,search_data)
    result_strings.append(full_result_string)


##### Extrinsic
import matplotlib.pyplot as plt
plt.close('all')  # 关闭所有现有 figure，避免残留
dir_string,dataset_strings = monod.preprocess.construct_batch(loom_filepaths, \
                                             transcriptome_filepath, \
                                             dataset_names, \
                                             attribute_names=attribute_names,\
                                             batch_location='/data/wuwq/noise/Monod_FLU/OUT', meta='nor',batch_id= 'B_Extrinsic',\
                                             n_genes=1000) # 可以改基因数： 250 or 5,000  → 1000足够了
n_datasets = len(dataset_names)
lb = [-1.0, -1.8, -1.8 ]
ub = [4.2, 2.5, 3.5]
grid = [6,7]
# 参数选择：
# biological_model = {'Bursty','Constitutive','Extrinsic','CIR'}       sequencing_model = {'None','Bernoulli','Poisson'}
result_strings = []
for i in range(n_datasets):
    fitmodel = monod.cme_toolbox.CMEModel('Extrinsic','Poisson')  # 改模型选择 Bursty or Extrinsic
    inference_parameters = monod.inference.InferenceParameters(lb,ub,[-8, -3],[-5, 0],grid,\
                dataset_strings[i],fitmodel,use_lengths = True,
                gradient_params = {'max_iterations':5,'init_pattern':'moments','num_restarts':1})
    search_data = monod.extract_data.extract_data(loom_filepaths[i], transcriptome_filepath, dataset_names[i],
                        dataset_strings[i], dir_string, dataset_attr_names=attribute_names)
    full_result_string = inference_parameters.fit_all_grid_points(1,search_data)
    result_strings.append(full_result_string)